# Ablation Experiments

The brief asks for "hyperparameter tuning" to be part of the documented methodology,
not just the final configs. The five main notebooks each report one settled
configuration; this one actually tests the reasoning behind three of those decisions
by running a real alternative and recording what happened — not what we assumed would
happen.

Each one follows the same loop: decision -> alternative -> real result -> what it means
-> does it change anything? Results are saved under `reports/results/ablations/` so
they don't overwrite the main five-approach results used in
`notebooks/results_comparison.ipynb`.

| Experiment | Change | Why test it |
|---|---|---|
| A | TF-IDF n-gram range: unigrams only vs. unigrams+bigrams | Finding #11 showed sentiment-bearing bigrams (`cause autism`, `vaccinate kids`) — does the baseline actually need them? |
| B | BiLSTM: class-weighted vs. unweighted cross-entropy | Finding #1 (10.4% Negative) is why we added class weighting everywhere — is it actually earning its keep? |
| C | CNN+BiGRU+Attention: `max_seq_len` 24 vs. 32 | Finding #4 put the 95th-percentile word length at 24; we went with 32 for margin — does the cheaper, EDA-exact budget do just as well, or does it lose information? |
| D | Full preprocessing pipeline vs. raw, uncleaned text | Every result in this project assumes cleaning helps -- that's never been tested directly |
| E | BiLSTM: 3 different random seeds | The main result is one seed -- how much of it is just luck of the draw? |
| F | TF-IDF+LogReg: 5-fold CV vs. the single 85/15 split | The main results all come from one split -- how much does the split itself matter? |


In [1]:
import sys
sys.path.append('../src')
import json
from pathlib import Path

import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

from preprocessing import LABEL_TO_IDX
from text_dataset import Vocab, TweetDataset
from models import BiLSTMClassifier, CNNBiGRUAttention
from train_utils import compute_class_weights, train_model, predict
from eval_utils import set_seed

ABLATION_DIR = Path('../reports/results/ablations')
ABLATION_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv('../data/processed/train_split.csv')
val_df = pd.read_csv('../data/processed/val_split.csv')
train_texts = train_df['clean_text'].tolist()
val_texts = val_df['clean_text'].tolist()
train_labels = train_df['label'].map(LABEL_TO_IDX).tolist()
val_labels = val_df['label'].map(LABEL_TO_IDX).tolist()
y_train_arr = train_df['label'].map(LABEL_TO_IDX).values
y_val_arr = val_df['label'].map(LABEL_TO_IDX).values

set_seed(42)
torch.set_num_threads(4)
device = 'cpu'

experiment_log = []


## Experiment A: TF-IDF n-gram range (unigrams only vs. 1-2 grams)

Decision being tested: `notebooks/baselines.ipynb` uses `ngram_range=(1, 2)`.
Alternative: `ngram_range=(1, 1)`, everything else held constant (`min_df=2`,
`max_features=20000`, same Logistic Regression setup).


In [2]:
results_a = {}
for label, ngram_range in [('unigram_only', (1, 1)), ('unigram_bigram', (1, 2))]:
    vec = TfidfVectorizer(ngram_range=ngram_range, min_df=2, max_features=20000, sublinear_tf=True)
    X_tr = vec.fit_transform(train_texts)
    X_va = vec.transform(val_texts)
    clf = LogisticRegression(max_iter=2000, class_weight='balanced', C=1.0, random_state=42)
    clf.fit(X_tr, y_train_arr)
    preds = clf.predict(X_va)
    macro_f1 = f1_score(y_val_arr, preds, average='macro')
    results_a[label] = {'vocab_size': len(vec.vocabulary_), 'macro_f1': macro_f1}
    print(f"{label}: vocab={len(vec.vocabulary_)}  macro_f1={macro_f1:.4f}")

with open(ABLATION_DIR / 'exp_A_ngram_range.json', 'w') as f:
    json.dump(results_a, f, indent=2)

delta_a = results_a['unigram_bigram']['macro_f1'] - results_a['unigram_only']['macro_f1']
print(f"\nDelta (bigrams - unigrams only): {delta_a:+.4f}")
experiment_log.append({
    'experiment': 'A', 'change': 'TF-IDF ngram_range (1,1) vs (1,2)',
    'reason': 'EDA finding #11: sentiment bigrams observed',
    'model': 'TFIDF+LogReg', 'result': f"unigram={results_a['unigram_only']['macro_f1']:.4f}, "
              f"1-2gram={results_a['unigram_bigram']['macro_f1']:.4f} (delta {delta_a:+.4f})",
})


unigram_only: vocab=5479  macro_f1=0.6592


unigram_bigram: vocab=18431  macro_f1=0.6747

Delta (bigrams - unigrams only): +0.0155


## Experiment B: BiLSTM class-weighted vs. unweighted loss

Decision being tested: `notebooks/bilstm.ipynb` uses class-weighted cross-entropy
(weights from `compute_class_weights`, based on finding #1). Alternative: plain
unweighted cross-entropy, same architecture and training budget otherwise. We use a
smaller epoch budget here (6, no early-stopping tuning) just to keep this fast — we
care about the relative effect of weighting, not matching the main result exactly.


In [3]:
MAX_LEN = 32
vocab_b = Vocab(train_texts, min_freq=2, max_size=20000)
train_ds_b = TweetDataset(train_texts, train_labels, vocab_b, max_len=MAX_LEN)
val_ds_b = TweetDataset(val_texts, val_labels, vocab_b, max_len=MAX_LEN)

real_weights = compute_class_weights(train_labels)
uniform_weights = torch.ones(3)

results_b = {}
for label, weights in [('unweighted', uniform_weights), ('class_weighted', real_weights)]:
    set_seed(42)
    model_b = BiLSTMClassifier(vocab_size=len(vocab_b), embed_dim=128, hidden_dim=128,
                                n_classes=3, pad_idx=vocab_b.stoi['<pad>'])
    model_b, _ = train_model(model_b, train_ds_b, val_ds_b, weights,
                              epochs=6, batch_size=64, lr=1e-3, patience=6, device=device)
    preds_b = predict(model_b, val_ds_b, device=device)
    macro_f1 = f1_score(val_labels, preds_b, average='macro')
    per_class_f1 = f1_score(val_labels, preds_b, average=None).tolist()
    results_b[label] = {'macro_f1': macro_f1, 'per_class_f1': per_class_f1}
    print(f"{label}: macro_f1={macro_f1:.4f}  per_class_f1(Neg,Neu,Pos)={[round(x,3) for x in per_class_f1]}")

with open(ABLATION_DIR / 'exp_B_class_weighting.json', 'w') as f:
    json.dump(results_b, f, indent=2)

delta_b = results_b['class_weighted']['macro_f1'] - results_b['unweighted']['macro_f1']
delta_neg = results_b['class_weighted']['per_class_f1'][0] - results_b['unweighted']['per_class_f1'][0]
print(f"\nDelta macro-F1 (weighted - unweighted): {delta_b:+.4f}")
print(f"Delta Negative-class F1 (weighted - unweighted): {delta_neg:+.4f}")
experiment_log.append({
    'experiment': 'B', 'change': 'BiLSTM class-weighted vs unweighted CE loss',
    'reason': 'EDA finding #1: Negative class only 10.4% of data',
    'model': 'BiLSTM', 'result': f"unweighted={results_b['unweighted']['macro_f1']:.4f}, "
              f"weighted={results_b['class_weighted']['macro_f1']:.4f} (delta {delta_b:+.4f}; "
              f"Negative-F1 delta {delta_neg:+.4f})",
})


Epoch 1/6 | train_loss=0.8842 val_loss=0.7834 val_macro_f1=0.4532 (27.6s)


Epoch 2/6 | train_loss=0.7304 val_loss=0.7376 val_macro_f1=0.4919 (28.5s)


Epoch 3/6 | train_loss=0.6471 val_loss=0.7202 val_macro_f1=0.5395 (29.1s)


Epoch 4/6 | train_loss=0.5603 val_loss=0.7388 val_macro_f1=0.5818 (20.3s)


Epoch 5/6 | train_loss=0.4667 val_loss=0.7735 val_macro_f1=0.6258 (22.3s)


...

Epoch 3/6 | train_loss=0.7528 val_loss=0.8655 val_macro_f1=0.5575 (13.7s)


Epoch 4/6 | train_loss=0.6435 val_loss=0.8841 val_macro_f1=0.6127 (17.1s)


Epoch 5/6 | train_loss=0.5157 val_loss=0.9659 val_macro_f1=0.5924 (14.4s)


Epoch 6/6 | train_loss=0.4280 val_loss=1.0655 val_macro_f1=0.6255 (20.8s)


class_weighted: macro_f1=0.6255  per_class_f1(Neg,Neu,Pos)=[0.432, 0.765, 0.679]

Delta macro-F1 (weighted - unweighted): -0.0057
Delta Negative-class F1 (weighted - unweighted): +0.0130


## Experiment C: CNN+BiGRU+Attention max_seq_len (24 vs. 32)

Decision being tested: `notebooks/cnn_bigru_attention.ipynb` uses `max_seq_len=32`.
Alternative: `max_seq_len=24`, the exact 95th-percentile word length from finding #4.
Same architecture and training budget otherwise.


In [4]:
results_c = {}
for max_len in [24, 32]:
    vocab_c = Vocab(train_texts, min_freq=2, max_size=20000)
    train_ds_c = TweetDataset(train_texts, train_labels, vocab_c, max_len=max_len)
    val_ds_c = TweetDataset(val_texts, val_labels, vocab_c, max_len=max_len)
    class_weights_c = compute_class_weights(train_labels)

    set_seed(42)
    model_c = CNNBiGRUAttention(vocab_size=len(vocab_c), embed_dim=128, n_filters=64,
                                 filter_sizes=(2, 3, 4), gru_hidden=96, n_classes=3,
                                 pad_idx=vocab_c.stoi['<pad>'])
    model_c, _ = train_model(model_c, train_ds_c, val_ds_c, class_weights_c,
                              epochs=8, batch_size=64, lr=1e-3, patience=4, device=device)
    preds_c = predict(model_c, val_ds_c, device=device)
    macro_f1 = f1_score(val_labels, preds_c, average='macro')
    results_c[f'max_len_{max_len}'] = {'macro_f1': macro_f1}
    print(f"max_len={max_len}: macro_f1={macro_f1:.4f}")

with open(ABLATION_DIR / 'exp_C_max_seq_len.json', 'w') as f:
    json.dump(results_c, f, indent=2)

delta_c = results_c['max_len_32']['macro_f1'] - results_c['max_len_24']['macro_f1']
print(f"\nDelta macro-F1 (len32 - len24): {delta_c:+.4f}")
experiment_log.append({
    'experiment': 'C', 'change': 'CNN+BiGRU+Attention max_seq_len 24 vs 32',
    'reason': 'EDA finding #4: 95th-percentile word length is 24',
    'model': 'CNN+BiGRU+Attention', 'result': f"len24={results_c['max_len_24']['macro_f1']:.4f}, "
              f"len32={results_c['max_len_32']['macro_f1']:.4f} (delta {delta_c:+.4f})",
})


Epoch 1/8 | train_loss=0.9790 val_loss=0.9135 val_macro_f1=0.4642 (16.9s)


Epoch 2/8 | train_loss=0.8199 val_loss=0.8562 val_macro_f1=0.5449 (15.1s)


Epoch 3/8 | train_loss=0.6780 val_loss=0.8670 val_macro_f1=0.5522 (15.6s)


Epoch 4/8 | train_loss=0.5352 val_loss=0.9777 val_macro_f1=0.6134 (15.1s)


Epoch 5/8 | train_loss=0.3771 val_loss=1.2656 val_macro_f1=0.6125 (15.2s)


...

Epoch 5/8 | train_loss=0.3808 val_loss=1.2128 val_macro_f1=0.6228 (21.4s)


Epoch 6/8 | train_loss=0.2543 val_loss=1.2589 val_macro_f1=0.6009 (21.8s)


Epoch 7/8 | train_loss=0.1657 val_loss=1.6636 val_macro_f1=0.6023 (22.9s)


Epoch 8/8 | train_loss=0.1509 val_loss=1.8621 val_macro_f1=0.5936 (29.0s)


max_len=32: macro_f1=0.6228

Delta macro-F1 (len32 - len24): +0.0093


## Experiment D: Does our preprocessing pipeline actually matter?

Design decision being tested: `src/preprocessing.py` (HTML-unescape, `<user>`/`<url>`
normalisation, elongation collapsing) is applied before every model, on the assumption
that it helps. Alternative: skip cleaning entirely and feed raw `safe_text` straight
into the same TF-IDF + Logistic Regression pipeline. Same train/val rows, same
vectorizer settings, same classifier — the only thing that changes is whether the text
was cleaned first.


In [5]:
results_d = {}
for label, col in [('raw_uncleaned', 'safe_text'), ('our_preprocessing', 'clean_text')]:
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000, sublinear_tf=True)
    X_tr = vec.fit_transform(train_df[col].fillna(''))
    X_va = vec.transform(val_df[col].fillna(''))
    clf = LogisticRegression(max_iter=2000, class_weight='balanced', C=1.0, random_state=42)
    clf.fit(X_tr, y_train_arr)
    preds = clf.predict(X_va)
    macro_f1 = f1_score(y_val_arr, preds, average='macro')
    results_d[label] = {'vocab_size': len(vec.vocabulary_), 'macro_f1': macro_f1}
    print(f"{label}: vocab={len(vec.vocabulary_)}  macro_f1={macro_f1:.4f}")

with open(ABLATION_DIR / 'exp_D_preprocessing.json', 'w') as f:
    json.dump(results_d, f, indent=2)

delta_d = results_d['our_preprocessing']['macro_f1'] - results_d['raw_uncleaned']['macro_f1']
print(f"\nDelta (our preprocessing - raw): {delta_d:+.4f}")
experiment_log.append({
    'experiment': 'D', 'change': 'Full preprocessing pipeline vs. raw safe_text',
    'reason': 'Every result in this project assumes preprocessing helps -- never tested directly',
    'model': 'TFIDF+LogReg', 'result': f"raw={results_d['raw_uncleaned']['macro_f1']:.4f}, "
              f"cleaned={results_d['our_preprocessing']['macro_f1']:.4f} (delta {delta_d:+.4f})",
})


raw_uncleaned: vocab=18505  macro_f1=0.6755


our_preprocessing: vocab=18431  macro_f1=0.6747

Delta (our preprocessing - raw): -0.0008


## Experiment E: How much does the random seed alone move BiLSTM's result?

Every main result in this project is a single run at `seed=42`. Here we train the
BiLSTM three times with different seeds (42, 123, 2024), everything else identical, to
get a real estimate of run-to-run variance -- not to pick a "best" seed.


In [6]:
results_e = {}
MAX_LEN = 32
for seed in [42, 123, 2024]:
    set_seed(seed)
    vocab_e = Vocab(train_texts, min_freq=2, max_size=20000)
    train_ds_e = TweetDataset(train_texts, train_labels, vocab_e, max_len=MAX_LEN)
    val_ds_e = TweetDataset(val_texts, val_labels, vocab_e, max_len=MAX_LEN)
    class_weights_e = compute_class_weights(train_labels)

    model_e = BiLSTMClassifier(vocab_size=len(vocab_e), embed_dim=128, hidden_dim=128,
                                n_classes=3, pad_idx=vocab_e.stoi['<pad>'])
    model_e, _ = train_model(model_e, train_ds_e, val_ds_e, class_weights_e,
                              epochs=12, batch_size=64, lr=1e-3, patience=3, device=device)
    preds_e = predict(model_e, val_ds_e, device=device)
    macro_f1 = f1_score(val_labels, preds_e, average='macro')
    results_e[f'seed_{seed}'] = {'macro_f1': macro_f1}
    print(f"seed={seed}: macro_f1={macro_f1:.4f}")

import numpy as np
seed_f1s = [v['macro_f1'] for v in results_e.values()]
results_e['summary'] = {'mean': float(np.mean(seed_f1s)), 'std': float(np.std(seed_f1s)),
                         'min': float(np.min(seed_f1s)), 'max': float(np.max(seed_f1s))}
print(f"\nAcross 3 seeds: mean={results_e['summary']['mean']:.4f} "
      f"std={results_e['summary']['std']:.4f} "
      f"range=[{results_e['summary']['min']:.4f}, {results_e['summary']['max']:.4f}]")

with open(ABLATION_DIR / 'exp_E_seed_variance.json', 'w') as f:
    json.dump(results_e, f, indent=2)

experiment_log.append({
    'experiment': 'E', 'change': 'BiLSTM trained with 3 different random seeds',
    'reason': 'The main result is a single seed=42 run -- how much does that alone explain?',
    'model': 'BiLSTM', 'result': f"mean={results_e['summary']['mean']:.4f} "
              f"std={results_e['summary']['std']:.4f} "
              f"range=[{results_e['summary']['min']:.4f}, {results_e['summary']['max']:.4f}]",
})


Epoch 1/12 | train_loss=1.0192 val_loss=0.9323 val_macro_f1=0.5122 (16.5s)


Epoch 2/12 | train_loss=0.8677 val_loss=0.8926 val_macro_f1=0.5639 (17.5s)


Epoch 3/12 | train_loss=0.7528 val_loss=0.8655 val_macro_f1=0.5575 (16.1s)


Epoch 4/12 | train_loss=0.6435 val_loss=0.8841 val_macro_f1=0.6127 (15.5s)


Epoch 5/12 | train_loss=0.5157 val_loss=0.9659 val_macro_f1=0.5924 (16.6s)


...

Epoch 9/12 | train_loss=0.1853 val_loss=1.5164 val_macro_f1=0.6202 (11.5s)


Epoch 10/12 | train_loss=0.1371 val_loss=1.6446 val_macro_f1=0.5905 (11.6s)


Epoch 11/12 | train_loss=0.1693 val_loss=1.5673 val_macro_f1=0.6004 (12.2s)


Epoch 12/12 | train_loss=0.1032 val_loss=1.8134 val_macro_f1=0.5996 (13.1s)
Early stopping at epoch 12 (no improvement for 3 epochs).


seed=2024: macro_f1=0.6202

Across 3 seeds: mean=0.6233 std=0.0057 range=[0.6184, 0.6314]


## Experiment F: 5-fold cross-validation for the best baseline

The main results all come from one stratified 85/15 split. Here we run 5-fold
stratified cross-validation for TF-IDF + Logistic Regression (the overall best
approach) on the full cleaned, de-duplicated training data, to see how much the
macro-F1 number actually moves across different splits.


In [7]:
from sklearn.model_selection import StratifiedKFold

full_df = pd.concat([train_df, val_df], ignore_index=True)
X_full_text = full_df['clean_text'].fillna('')
y_full = full_df['label'].map(LABEL_TO_IDX).values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full_text, y_full)):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000, sublinear_tf=True)
    X_tr = vec.fit_transform(X_full_text.iloc[tr_idx])
    X_va = vec.transform(X_full_text.iloc[va_idx])
    clf = LogisticRegression(max_iter=2000, class_weight='balanced', C=1.0, random_state=42)
    clf.fit(X_tr, y_full[tr_idx])
    preds = clf.predict(X_va)
    fold_f1 = f1_score(y_full[va_idx], preds, average='macro')
    fold_scores.append(fold_f1)
    print(f"fold {fold}: macro_f1={fold_f1:.4f}")

results_f = {'fold_scores': fold_scores, 'mean': float(np.mean(fold_scores)),
             'std': float(np.std(fold_scores))}
print(f"\n5-fold CV: mean={results_f['mean']:.4f} std={results_f['std']:.4f}")
print(f"Single-split result from notebooks/baselines.ipynb: 0.6747")

with open(ABLATION_DIR / 'exp_F_kfold_cv.json', 'w') as f:
    json.dump(results_f, f, indent=2)

experiment_log.append({
    'experiment': 'F', 'change': '5-fold stratified CV vs. single 85/15 split',
    'reason': 'The main result is a single split -- how much does macro-F1 vary across different ones?',
    'model': 'TFIDF+LogReg', 'result': f"CV mean={results_f['mean']:.4f} std={results_f['std']:.4f} "
              f"(single-split result was 0.6747)",
})


fold 0: macro_f1=0.6735


fold 1: macro_f1=0.6569


fold 2: macro_f1=0.6507


fold 3: macro_f1=0.6704


fold 4: macro_f1=0.6473

5-fold CV: mean=0.6598 std=0.0105
Single-split result from notebooks/baselines.ipynb: 0.6747


## Experiment G: BiLSTM with pretrained GloVe embeddings vs. from-scratch

The literature review (`related_work.md`, Section 1) cites GloVe (Pennington et al.,
2014) as the standard alternative to learning embeddings from scratch, but the main
BiLSTM result never actually used it -- a gap flagged honestly in Section 6.4 of the
report. This experiment closes it: same BiLSTM architecture and training budget as the
main result, the only difference is whether the embedding layer starts from GloVe
Twitter vectors (100d, pretrained on 2B tweets -- a good domain match for this
dataset) or from random initialization. Embeddings are fine-tuned during training in
both cases (not frozen), so this isolates the effect of the *starting point*, not
whether the embeddings can adapt.


In [8]:
from embeddings import load_glove, build_embedding_matrix

GLOVE_PATH = '../data/glove.twitter.27B.100d.txt'
GLOVE_DIM = 100

glove = load_glove(GLOVE_PATH)
print(f'Loaded {len(glove)} GloVe vectors')

vocab_g = Vocab(train_texts, min_freq=2, max_size=20000)
embedding_matrix, hits = build_embedding_matrix(vocab_g, glove, GLOVE_DIM)
print(f'Vocab coverage: {hits}/{len(vocab_g)} words found in GloVe ({hits/len(vocab_g):.1%})')

train_ds_g = TweetDataset(train_texts, train_labels, vocab_g, max_len=32)
val_ds_g = TweetDataset(val_texts, val_labels, vocab_g, max_len=32)
class_weights_g = compute_class_weights(train_labels)

results_g = {}
for label, use_pretrained in [('from_scratch', False), ('glove_pretrained', True)]:
    set_seed(42)
    model_g = BiLSTMClassifier(
        vocab_size=len(vocab_g), embed_dim=GLOVE_DIM, hidden_dim=128, n_classes=3,
        pad_idx=vocab_g.stoi['<pad>'],
        pretrained_embeddings=embedding_matrix if use_pretrained else None,
    )
    model_g, _ = train_model(model_g, train_ds_g, val_ds_g, class_weights_g,
                              epochs=12, batch_size=64, lr=1e-3, patience=3, device=device)
    preds_g = predict(model_g, val_ds_g, device=device)
    macro_f1 = f1_score(val_labels, preds_g, average='macro')
    results_g[label] = {'macro_f1': macro_f1}
    print(f'{label}: macro_f1={macro_f1:.4f}')

with open(ABLATION_DIR / 'exp_G_glove_embeddings.json', 'w') as f:
    json.dump({**results_g, 'vocab_coverage': hits / len(vocab_g)}, f, indent=2)

delta_g = results_g['glove_pretrained']['macro_f1'] - results_g['from_scratch']['macro_f1']
print(f"\nDelta (GloVe - from-scratch): {delta_g:+.4f}")
experiment_log.append({
    'experiment': 'G', 'change': 'BiLSTM embeddings: GloVe-pretrained vs. from-scratch',
    'reason': 'GloVe is cited in related_work.md but was never actually used -- closing that gap',
    'model': 'BiLSTM', 'result': f"from_scratch={results_g['from_scratch']['macro_f1']:.4f}, "
              f"glove={results_g['glove_pretrained']['macro_f1']:.4f} (delta {delta_g:+.4f})",
})


Loaded 1193514 GloVe vectors


Vocab coverage: 4425/6991 words found in GloVe (63.3%)


Epoch 1/12 | train_loss=1.0348 val_loss=0.9642 val_macro_f1=0.5092 (9.5s)


Epoch 2/12 | train_loss=0.8933 val_loss=0.8888 val_macro_f1=0.5106 (9.4s)


Epoch 3/12 | train_loss=0.7796 val_loss=0.8373 val_macro_f1=0.5911 (10.6s)


...

Epoch 5/12 | train_loss=0.4912 val_loss=0.9602 val_macro_f1=0.6404 (11.1s)


Epoch 6/12 | train_loss=0.3887 val_loss=0.8946 val_macro_f1=0.6304 (10.7s)


Epoch 7/12 | train_loss=0.3056 val_loss=1.1520 val_macro_f1=0.6398 (10.6s)


Epoch 8/12 | train_loss=0.2385 val_loss=1.2392 val_macro_f1=0.6268 (11.0s)
Early stopping at epoch 8 (no improvement for 3 epochs).


glove_pretrained: macro_f1=0.6404

Delta (GloVe - from-scratch): -0.0016


## Experiment tracking summary

The interpretation and next-decision columns below are filled in after looking at the
three real results above — nothing here is invented, it's copied straight from the
printed output and the JSON files in `reports/results/ablations/`.


In [9]:
log_df = pd.DataFrame(experiment_log)
log_df


,experiment,change,reason,model,result
0,A,"TF-IDF ngram_range (1,1) vs (1,2)",EDA finding #11: sentiment bigrams observed,TFIDF+LogReg,"unigram=0.6592, 1-2gram=0.6747 (delta +0.0155)"
1,B,BiLSTM class-weighted vs unweighted CE loss,EDA finding #1: Negative class only 10.4% of data,BiLSTM,"unweighted=0.6312, weighted=0.6255 (delta -0.0..."
2,C,CNN+BiGRU+Attention max_seq_len 24 vs 32,EDA finding #4: 95th-percentile word length is 24,CNN+BiGRU+Attention,"len24=0.6134, len32=0.6228 (delta +0.0093)"
3,D,Full preprocessing pipeline vs. raw safe_text,Every result in this project assumes preproces...,TFIDF+LogReg,"raw=0.6755, cleaned=0.6747 (delta -0.0008)"
4,E,BiLSTM trained with 3 different random seeds,The main result is a single seed=42 run -- how...,BiLSTM,"mean=0.6233 std=0.0057 range=[0.6184, 0.6314]"
5,F,5-fold stratified CV vs. single 85/15 split,The main result is a single split -- how much ...,TFIDF+LogReg,CV mean=0.6598 std=0.0105 (single-split result...
6,G,BiLSTM embeddings: GloVe-pretrained vs. from-s...,GloVe is cited in related_work.md but was neve...,BiLSTM,"from_scratch=0.6420, glove=0.6404 (delta -0.0016)"


## Interpretation

What actually happened (see the printed output and `reports/results/ablations/*.json`):

- Experiment A (n-grams): bigrams help. Unigram-only macro-F1 is 0.6592; 1-2gram gets
  0.6747 (+0.0155). Finding #11's sentiment bigrams (`cause autism`) really do translate
  into a measurable gain, so `ngram_range=(1,2)` in the main result earns its place.
- Experiment B (class weighting): a genuine trade-off, not a clean win. Unweighted
  macro-F1 is 0.6312 (Negative-F1 0.419); weighted gets 0.6255 (Negative-F1 0.432).
  Weighting does what it's supposed to -- Negative-class F1 goes up by 0.013, directly
  addressing finding #1 -- but it costs enough on Neutral/Positive that overall macro-F1
  drops slightly (-0.0057).
- Experiment C (sequence length): the extra margin past the exact percentile helps.
  `max_len=24` gets macro-F1 0.6134; `max_len=32` gets 0.6228 (+0.0093). Cutting off
  right at the 95th percentile loses enough from the remaining 5% of longer tweets to
  cost real accuracy.
- Experiment D (preprocessing): barely matters for TF-IDF. Raw, uncleaned text gets
  macro-F1 0.6755; our full preprocessing pipeline gets 0.6747 -- a difference small
  enough to be noise. TF-IDF tokenizes on whitespace regardless of HTML entities or
  elongated letters, so cleaning mostly matters for the word-level and subword models,
  not this one.
- Experiment E (seed variance): BiLSTM's macro-F1 across seeds 42/123/2024 is
  0.6314/0.6184/0.6202 -- mean 0.6233, std 0.0057. Our main reported result (seed=42)
  sits at the top of this range, not the middle.
- Experiment F (5-fold CV): TF-IDF+LogReg's macro-F1 across 5 folds averages 0.6598
  (std 0.0105), versus the single-split result of 0.6747 used everywhere else in the
  report. The split we happened to use is about 0.015 more favourable than average.
- Experiment G (GloVe vs. from-scratch): the one real surprise. GloVe-Twitter 100d
  embeddings score 0.6404 macro-F1 for BiLSTM, barely different from -- and slightly
  below -- the 0.6420 from training embeddings from scratch at the same dimension.
  Vocabulary coverage is only 63.3%, so pretraining only gives a head start on about
  two-thirds of the model's inputs, which doesn't appear to be enough to help here.

Taken together: A and C validate design choices we'd already made; B shows one of
them is a real trade-off, not a free win; D, E, and F show how much of the headline
numbers is actual signal versus preprocessing/seed/split noise; and G closes the
GloVe gap flagged in `related_work.md` with a genuine (if unexpected) negative
result. All seven show up in `reports/report.md`, Section 4.8.
